# Feature Engineering

Feature engineering is the process of converting raw data into meaningful features that better represent the underlying problem to predictive models. It involves selecting, transforming, and creating features to improve model performance.

## Why Feature Engineering?
- Makes learning faster and more stable
- Improves model accuracy
- Reduces overfitting
- Helps algorithms converge faster
- Makes patterns more visible to models

## Hands-on Implementation

### Step 1: Load Cleaned Dataset
Loading the preprocessed housing dataset from the previous stage.

In [2]:
import pandas as pd
import numpy as np

try:
    df = pd.read_csv("../Data/Housing_cleaned.csv")
    print("df loaded successfully:\n", df.head())
except FileNotFoundError:
    print("File not found! Please check the file path.")
    exit()
except Exception as e:
    print("Error while loading dataset:", str(e))
    exit()

df loaded successfully:
      price  area  bedrooms  bathrooms  stories mainroad  parking  \
0  9100000  6000         4          1        2      yes        2   
1  9100000  6600         4          2        2      yes        1   
2  8890000  4600         3          2        2      yes        2   
3  8855000  6420         3          2        2      yes        1   
4  8750000  4320         3          1        2      yes        2   

  furnishingstatus  
0   semi-furnished  
1      unfurnished  
2        furnished  
3   semi-furnished  
4   semi-furnished  


### Common Feature Engineering Techniques

#### 1. Normalization (Min-Max Scaling)
Converts numerical values into a standard range, typically 0 to 1.

**Example:**
- 500 → 0.0
- 1000 → 0.5
- 1500 → 1.0

**Benefits:**
- Keeps patterns same
- Improves gradient descent convergence
- Prevents features with larger scales from dominating

#### 2. Standardization (Z-Score Scaling)
Transforms data to have mean = 0 and standard deviation = 1.

**Benefits:**
- Handles outliers better than min-max scaling
- Suitable for algorithms that assume normally distributed data
- Preserves the shape of the original distribution

#### 3. Binning
Converts continuous numerical values into discrete ranges or bins.

**Example:**
- 0 - 600 → small
- 600 - 1200 → medium
- 1200+ → large

**Benefits:**
- Reduces noise in data
- Handles outliers naturally
- Used when exact values are not important

#### 4. One-Hot Encoding
Converts categorical variables into binary columns.

**Benefits:**
- Makes categorical data usable for machine learning models
- No ordinal relationship assumed between categories
- Works well with both tree-based and linear models

#### 5. Log Transformation
Applies logarithmic transformation to numerical features.

**Benefits:**
- Handles skewed distributions
- Makes data more normally distributed
- Reduces the effect of extreme values

### Step 2: Normalization (Min-Max Scaling)
Normalization is the process of transforming numerical features to a similar scale, typically between 0 and 1, so that machine learning models can train faster and more accurately.

## Formula
X_scaled = (X - X_min) / (X_max - X_min)

Where:
- X_min = minimum value in the feature
- X_max = maximum value in the feature

**Note:** Categorical columns (mainroad, furnishingstatus) are dropped for now. They cannot be directly scaled as they contain string values. We will handle categorical variables later using One-Hot Encoding.

In [3]:
# Splitting features and target
X = df.drop(["price", "mainroad", "furnishingstatus"], axis=1)
Y = df["price"]

print("Features (X) shape:", X.shape)
print("Features columns:", X.columns.tolist())
print("Target (Y) shape:", Y.shape)

Features (X) shape: (463, 5)
Features columns: ['area', 'bedrooms', 'bathrooms', 'stories', 'parking']
Target (Y) shape: (463,)


In [4]:
# Applying Min-Max Scaling
X_min = X.min()
X_max = X.max()
X_scaled = (X - X_min) / (X_max - X_min)

print("Min-Max scaling done")
print(X_scaled.head())

Min-Max scaling done
       area  bedrooms  bathrooms  stories  parking
0  0.491525  1.000000        0.0      0.5      1.0
1  0.559322  1.000000        0.5      0.5      0.5
2  0.333333  0.666667        0.5      0.5      1.0
3  0.538983  0.666667        0.5      0.5      0.5
4  0.301695  0.666667        0.0      0.5      1.0


### Step 3: Standardization (Z-Score Scaling)
Standardization transforms data to have a mean of 0 and a standard deviation of 1. This technique is useful when the data follows a normal distribution and outliers are present.

## Formula
Z = (X - μ) / σ

Where:
- μ (mu) = mean of the feature
- σ (sigma) = standard deviation of the feature

In [5]:
# Applying Z-Score Standardization
mean = np.mean(X)
sigma = X.std()
Z_score = (X - mean) / sigma

print("Z score done")
print(Z_score.head())

Z score done
       area     bedrooms    bathrooms      stories      parking
0  2.810330 -1442.612812 -2166.170250 -1519.495652 -1227.537769
1  3.144533 -1442.612812 -2163.898614 -1519.495652 -1228.826424
2  2.030522 -1444.130436 -2163.898614 -1519.495652 -1227.537769
3  3.044272 -1444.130436 -2163.898614 -1519.495652 -1228.826424
4  1.874561 -1444.130436 -2166.170250 -1519.495652 -1227.537769


### Step 4: Log Transformation

## Formula:
X' = ln(X)

Log transformation is used to handle skewed data and reduce the effect of extreme values. Natural logarithm is commonly used.

In [6]:
import numpy as np
X_log = np.log(X)
print("Log transformation done")
print(X_log.head())

Log transformation done
       area  bedrooms  bathrooms   stories   parking
0  8.699515  1.386294   0.000000  0.693147  0.693147
1  8.794825  1.386294   0.693147  0.693147  0.000000
2  8.433812  1.098612   0.693147  0.693147  0.693147
3  8.767173  1.098612   0.693147  0.693147  0.000000
4  8.371011  1.098612   0.000000  0.693147  0.693147


C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)


### Step 5: Clipping (Not True Normalization)

Clipping is a technique used to limit extreme values in the data. It helps reduce the effects of outliers by capping values at certain thresholds.

## Formula:
X_clipped = max(lower_bound, min(upper_bound, X))

## Benefits:
- Reduces effect of extreme outliers
- Prevents distortion of model training
- Works well with tree-based models
- Useful when data has extreme values

In [7]:
# Defining bounds for clipping
# For area: 3000 to 12000
# For bedrooms: 1 to 6
# For bathrooms: 1 to 4
# For stories: 1 to 4
# For parking: 0 to 3

bounds = {
    'area': (3000, 12000),
    'bedrooms': (1, 6),
    'bathrooms': (1, 4),
    'stories': (1, 4),
    'parking': (0, 3)
}

X_clipped = X.copy()

for col, (lower, upper) in bounds.items():
    X_clipped[col] = X_clipped[col].clip(lower=lower, upper=upper)

print("Clipping done")
print("Clipped data (first 5 rows):")
print(X_clipped.head())

Clipping done
Clipped data (first 5 rows):
   area  bedrooms  bathrooms  stories  parking
0  6000         4          1        2        2
1  6600         4          2        2        1
2  4600         3          2        2        2
3  6420         3          2        2        1
4  4320         3          1        2        2


### Summary of Scaling Techniques - Part 1

#### Techniques Covered:

| Technique | Formula | When to Use |
|-----------|---------|-------------|
| Min-Max Scaling | (X - X_min) / (X_max - X_min) | When data is not normally distributed, needs bounded range [0,1] |
| Z-Score Scaling | (X - μ) / σ | When data follows normal distribution, needs mean=0, std=1 |
| Log Transformation | ln(X) | When data is skewed, to reduce effect of extreme values |
| Clipping | max(lower, min(upper, X)) | To limit extreme outliers without changing internal structure |

#### Observations:
- Min-Max scaling preserves original distribution shape
- Z-Score scaling centers data around 0
- Log transformation reduces right-skewness
- Clipping only affects extreme values

---

## KHTAM

Part 1 (Scaling) is complete. Next part will focus on Binning and other techniques.

---
End of Feature Engineering - Part 1